# Chapitre 18 · Apprendre des préférences (exercice)

Notebook **à trous**. Tu complètes les `# TODO(toi)` puis tu valides avec les `assert`.
Tout ce qui précède le premier TODO s'exécute déjà : lance d'abord le ré-entraînement du GPT, puis remplis les trous.

Le corrigé complet est dans `solutions/partie_4_du_modele_a_lesprit/`.

## Setup

Tout tourne sur CPU, hors ligne, en quelques minutes.

In [ ]:
import copy
import math
import random
import statistics
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)
print("PyTorch", torch.__version__)

## 1. Le point de départ : le GPT des fables

On reconstruit le GPT du chapitre 10 (mêmes classes, mêmes hyperparamètres) et on le ré-entraîne 1 500 pas sur les trente fables : ça suffit pour retrouver un modèle qui écrit du pseudo-français correct, notre « modèle qui obéit » du chapitre 17, version compacte et autonome. Si tu as encore le notebook du chapitre 17 sous la main, tu peux comparer : c'est le même moteur.

In [ ]:
corpus = """\
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêter
Quelque grain pour subsister
Jusqu'à la saison nouvelle.
Je vous paierai, lui dit-elle,
Avant l'oût, foi d'animal,
Intérêt et principal.
La fourmi n'est pas prêteuse :
C'est là son moindre défaut.
Que faisiez-vous au temps chaud ?
Dit-elle à cette emprunteuse. -
Nuit et jour à tout venant
Je chantais, ne vous déplaise. -
Vous chantiez, j'en suis fort aise !
Eh bien ! dansez maintenant.


LE CORBEAU ET LE RENARD
Maître corbeau, sur un arbre perché,
Tenait en son bec un fromage.
Maître renard, par l'odeur alléché,
Lui tint à peu près ce langage :
Hé ! bonjour, monsieur du corbeau.
Que vous êtes joli ! que vous me semblez beau !
Sans mentir, si votre ramage
Se rapporte à votre plumage,
Vous êtes le phénix des hôtes de ces bois.
À ces mots le corbeau ne se sent pas de joie ;
Et, pour montrer sa belle voix,
Il ouvre un large bec, laisse tomber sa proie.
Le renard s'en saisit, et dit : Mon bon monsieur,
Apprenez que tout flatteur
Vit aux dépens de celui qui l'écoute :
Cette leçon vaut bien un fromage, sans doute.
Le corbeau, honteux et confus,
Jura, mais un peu tard, qu'on ne l'y prendrait plus.


LA GRENOUILLE QUI SE VEUT FAIRE AUSSI GROSSE QUE LE BŒUF
Une grenouille vit un bœuf
Qui lui sembla de belle taille.
Elle, qui n'était pas grosse en tout comme un œuf,
Envieuse, s'étend, et s'enfle, et se travaille
Pour égaler l'animal en grosseur ;
Disant : Regardez bien, ma sœur ;
Est-ce assez ? dites-moi ; n'y suis-je point encore ? -
Nenni. - M'y voici donc ? - Point du tout. - M'y voilà ? -
Vous n'en approchez point. La chétive pécore
S'enfla si bien qu'elle creva.
Le monde est plein de gens qui ne sont pas plus sages :
Tout bourgeois veut bâtir comme les grands seigneurs,
Tout petit prince a des ambassadeurs,
Tout marquis veut avoir des pages.


LE LOUP ET LE CHIEN
Un loup n'avait que les os et la peau,
Tant les chiens faisaient bonne garde.
Ce loup rencontre un dogue aussi puissant que beau,
Gras, poli, qui s'était fourvoyé par mégarde.
L'attaquer, le mettre en quartiers,
Sire loup l'eût fait volontiers :
Mais il fallait livrer bataille ;
Et le mâtin était de taille
À se défendre hardiment.
Le loup donc l'aborde humblement,
Entre en propos, et lui fait compliment
Sur son embonpoint, qu'il admire.
Il ne tiendra qu'à vous, beau sire,
D'être aussi gras que moi, lui repartit le chien.
Quittez les bois, vous ferez bien :
Vos pareils y sont misérables,
Cancres, hères et pauvres diables,
Dont la condition est de mourir de faim.
Car, quoi ! rien d'assuré ! point de franche lippée !
Tout à la pointe de l'épée !
Suivez-moi, vous aurez un bien meilleur destin.
Le loup reprit : Que me faudra-t-il faire ?
Presque rien, dit le chien : donner la chasse aux gens
Portants bâtons, et mendiants ;
Flatter ceux du logis, à son maître complaire ;
Moyennant quoi votre salaire
Sera force reliefs de toutes les façons,
Os de poulets, os de pigeons ;
Sans parler de mainte caresse.
Le loup déjà se forge une félicité
Qui le fait pleurer de tendresse.
Chemin faisant il vit le cou du chien pelé.
Qu'est-ce là ? lui dit-il. - Rien. - Quoi ! rien ! - Peu de chose. -
Mais encor ? - Le collier dont je suis attaché
De ce que vous voyez est peut-être la cause.
Attaché ! dit le loup : vous ne courez donc pas
Où vous voulez ? - Pas toujours ; mais qu'importe ?
Il importe si bien, que de tous vos repas
Je ne veux en aucune sorte,
Et ne voudrais pas même à ce prix d'un trésor.
Cela dit, maître loup s'enfuit, et court encor.


LA BESACE
Jupiter dit un jour : Que tout ce qui respire
S'en vienne comparaître aux pieds de ma grandeur :
Si dans son composé quelqu'un trouve à redire,
Il peut le déclarer sans peur ;
Je mettrai remède à la chose.
Venez, singe ; parlez le premier, et pour cause :
Voyez ces animaux, faites comparaison
De leurs beautés avec les vôtres.
Êtes-vous satisfait ? - Moi, dit-il ; pourquoi non ?
N'ai-je pas quatre pieds aussi bien que les autres ?
Mon portrait jusqu'ici ne m'a rien reproché :
Mais pour mon frère l'ours, on ne l'a qu'ébauché ;
Jamais, s'il me veut croire, il ne se fera peindre.
L'ours venant là-dessus, on crut qu'il s'allait plaindre.
Tant s'en faut : de sa forme il se loua très-fort ;
Glosa sur l'éléphant, dit qu'on pourrait encor
Ajouter à sa queue, ôter à ses oreilles ;
Que c'était une masse informe et sans beauté.
L'éléphant étant écouté,
Tout sage qu'il était, dit des choses pareilles :
Il jugea qu'à son appétit
Dame baleine était trop grosse.
Dame fourmi trouva le ciron trop petit,
Se croyant, pour elle, un colosse.
Jupin les renvoya s'étant censurés tous,
Du reste, contents d'eux. Mais, parmi les plus fous,
Notre espèce excella ; car, tout ce que nous sommes,
Lynx envers nos pareils, et taupes envers nous,
Nous nous pardonnons tout, et rien aux autres hommes :
On se voit d'un autre œil qu'on ne voit son prochain.
Le fabricateur souverain
Nous créa besaciers tous de même manière,
Tant ceux du temps passé que du temps d'aujourd'hui :
Il fit pour nos défauts la poche de derrière,
Et celle de devant pour les défauts d'autrui.


LE LOUP ET L'AGNEAU
La raison du plus fort est toujours la meilleure :
Nous l'allons montrer tout à l'heure.
Un agneau se désaltérait
Dans le courant d'une onde pure.
Un loup survint à jeun, qui cherchait aventure,
Et que la faim en ces lieux attirait.
Qui te rend si hardi de troubler mon breuvage ?
Dit cet animal plein de rage :
Tu seras châtié de ta témérité.
Sire, répond l'agneau, que Votre Majesté
Ne se mette pas en colère ;
Mais plutôt qu'elle considère
Que je me vas désaltérant
Dans le courant,
Plus de vingt pas au-dessous d'elle ;
Et que, par conséquent, en aucune façon
Je ne puis troubler sa boisson.
Tu la troubles ! reprit cette bête cruelle ;
Et je sais que de moi tu médis l'an passé.
Comment l'aurais-je fait, si je n'étais pas né ?
Reprit l'agneau : je tette encore ma mère. -
Si ce n'est toi, c'est donc ton frère. -
Je n'en ai point. - C'est donc quelqu'un des tiens ;
Car vous ne m'épargnez guère,
Vous, vos bergers et vos chiens.
On me l'a dit : il faut que je me venge.
Là-dessus, au fond des forêts
Le loup l'emporte, et puis le mange,
Sans autre forme de procès.


LA MORT ET LE BÛCHERON
Un pauvre bucheron, tout couvert de ramée,
Sous le faix du fagot aussi bien que des ans,
Gémissant et courbé, marchait à pas pesants,
Et tâchait de gagner sa chaumine enfumée.
Enfin, n'en pouvant plus d'effort et de douleur,
Il met bas son fagot, il songe à son malheur.
Quel plaisir a-t-il eu depuis qu'il est au monde ?
En est-il un plus pauvre en la machine ronde ?
Point de pain quelquefois, et jamais de repos :
Sa femme, ses enfants, les soldats, les impôts,
Le créancier, et la corvée,
Lui font d'un malheureux la peinture achevée.
Il appelle la Mort. Elle vient sans tarder,
Lui demande ce qu'il faut faire.
C'est, dit-il, afin de m'aider
À recharger ce bois ; tu ne tarderas guère.
Le trépas vient tout guérir ;
Mais ne bougeons d'où nous sommes :
Plutôt souffrir que mourir,
C'est la devise des hommes.


LE RENARD ET LA CIGOGNE
Compère le renard se mit un jour en frais,
Et retint à dîner commère la cigogne.
Le régal fut petit et sans beaucoup d'apprêts :
Le galant, pour toute besogne,
Avait un brouet clair ; il vivait chichement.
Ce brouet fut par lui servi sur une assiette :
La cigogne au long bec n'en put attraper miette ;
Et le drôle eut lapé le tout en un moment.
Pour se venger de cette tromperie,
À quelque temps de là la cigogne le prie.
Volontiers, lui dit-il ; car avec mes amis
Je ne fais point cérémonie.
À l'heure dite, il courut au logis
De la cigogne son hôtesse ;
Loua très-fort sa politesse ;
Trouva le dîner cuit à point :
Bon appétit surtout ; renards n'en manquent point.
Il se réjouissait à l'odeur de la viande
Mise en menus morceaux, et qu'il croyait friande.
On servit, pour l'embarrasser,
En un vase à long col et d'étroite embouchure.
Le bec de la cigogne y pouvait bien passer ;
Mais le museau du sire était d'autre mesure.
Il lui fallut à jeun retourner au logis,
Honteux comme un renard qu'une poule aurait pris,
Serrant la queue, et portant bas l'oreille.
Trompeurs, c'est pour vous que j'écris :
Attendez-vous à la pareille.


LE CHÊNE ET LE ROSEAU
Le chêne un jour dit au roseau :
Vous avez bien sujet d'accuser la nature ;
Un roitelet pour vous est un pesant fardeau :
Le moindre vent qui d'aventure
Fait rider la face de l'eau,
Vous oblige à baisser la tête ;
Cependant que mon front, au Caucase pareil,
Non content d'arrêter les rayons du soleil,
Brave l'effort de la tempête.
Tout vous est aquilon, tout me semble zéphyr.
Encor si vous naissiez à l'abri du feuillage
Dont je couvre le voisinage,
Vous n'auriez pas tant à souffrir,
Je vous défendrais de l'orage :
Mais vous naissez le plus souvent
Sur les humides bords des royaumes du vent.
La nature envers vous me semble bien injuste.
Votre compassion, lui répondit l'arbuste,
Part d'un bon naturel ; mais quittez ce souci :
Les vents me sont moins qu'à vous redoutables ;
Je plie et ne romps pas. Vous avez jusqu'ici
Contre leurs coups épouvantables
Résisté sans courber le dos ;
Mais attendons la fin. Comme il disait ces mots,
Du bout de l'horizon accourt avec furie
Le plus terrible des enfants
Que le Nord eût portés jusque-là dans ses flancs.
L'arbre tient bon ; le roseau plie.
Le vent redouble ses efforts,
Et fait si bien qu'il déracine
Celui de qui la tête au ciel était voisine,
Et dont les pieds touchaient à l'empire des morts.


LE LION ET LE RAT
Il faut, autant qu'on peut, obliger tout le monde :
On a souvent besoin d'un plus petit que soi.
De cette vérité deux fables feront foi ;
Tant la chose en preuves abonde.
Entre les pattes d'un lion
Un rat sortit de terre assez à l'étourdie.
Le roi des animaux, en cette occasion,
Montra ce qu'il était, et lui donna la vie.
Ce bienfait ne fut pas perdu.
Quelqu'un aurait-il jamais cru
Qu'un lion d'un rat eût affaire ?
Cependant il advint qu'au sortir des forêts
Ce lion fut pris dans des rets,
Dont ses rugissements ne le purent défaire.
Sire rat accourut, et fit tant par ses dents
Qu'une maille rongée emporta tout l'ouvrage.
Patience et longueur de temps
Font plus que force ni que rage.


LA COLOMBE ET LA FOURMI
L'autre exemple est tiré d'animaux plus petits.
Le long d'un clair ruisseau buvait une colombe,
Quand sur l'eau se penchant une fourmis y tombe ;
Et dans cet océan on eût vu la fourmis
S'efforcer, mais en vain, de regagner la rive.
La colombe aussitôt usa de charité :
Un brin d'herbe dans l'eau par elle étant jeté,
Ce fut un promontoire où la fourmis arrive.
Elle se sauve. Et là-dessus
Passe un certain croquant qui marchait les pieds nus :
Ce croquant, par hasard, avait une arbalète.
Dès qu'il voit l'oiseau de Vénus,
Il le croit en son pot, et déjà lui fait fête.
Tandis qu'à le tuer mon villageois s'apprête,
La fourmi le pique au talon.
Le vilain retourne la tête :
La colombe l'entend, part, et tire de long.
Le souper du croquant avec elle s'envole :
Point de pigeon pour une obole.


LE LIÈVRE ET LA TORTUE
Rien ne sert de courir ; il faut partir à point :
Le lièvre et la tortue en sont un témoignage.
Gageons, dit celle-ci, que vous n'atteindrez point
Sitôt que moi ce but. Sitôt ! êtes-vous sage ?
Repartit l'animal léger :
Ma commère, il vous faut purger
Avec quatre grains d'ellébore.
- Sage ou non, je parie encore.
Ainsi fut fait ; et de tous deux
On mit près du but les enjeux.
Savoir quoi, ce n'est pas l'affaire,
Ni de quel juge l'on convint.
Notre lièvre n'avait que quatre pas à faire ;
J'entends de ceux qu'il fait lorsque, près d'être atteint,
Il s'éloigne des chiens, les renvoie aux calendes,
Et leur fait arpenter les landes.
Ayant, dis-je, du temps de reste pour brouter,
Pour dormir, et pour écouter
D'où vient le vent, il laisse la tortue
Aller son train de sénateur.
Elle part, elle s'évertue ;
Elle se hâte avec lenteur.
Lui cependant méprise une telle victoire,
Tient la gageure à peu de gloire,
Croit qu'il y va de son honneur
De partir tard. Il broute, il se repose ;
Il s'amuse à toute autre chose
Qu'à la gageure. À la fin, quand il vit
Que l'autre touchait presque au bout de la carrière,
Il partit comme un trait ; mais les élans qu'il fit
Furent vains : la tortue arriva la première.
Eh bien ! lui cria-t-elle, avais-je pas raison ?
De quoi vous sert votre vitesse ?
Moi l'emporter ! et que serait-ce
Si vous portiez une maison ?


LE RENARD ET LES RAISINS
Certain renard gascon, d'autres disent normand,
Mourant presque de faim, vit au haut d'une treille
Des raisins, mûrs apparemment,
Et couverts d'une peau vermeille.
Le galant en eût fait volontiers un repas ;
Mais comme il n'y pouvait atteindre :
Ils sont trop verts, dit-il, et bons pour des goujats.
Fit-il pas mieux que de se plaindre ?


LE HÉRON
Un jour, sur ses longs pieds, allait je ne sais où
Le héron au long bec emmanché d'un long cou :
Il côtoyait une rivière.
L'onde était transparente ainsi qu'aux plus beaux jours ;
Ma commère la carpe y faisait mille tours
Avec le brochet son compère.
Le héron en eût fait aisément son profit :
Tous approchaient du bord ; l'oiseau n'avait qu'à prendre.
Mais il crut mieux faire d'attendre
Qu'il eût un peu plus d'appétit :
Il vivait de régime, et mangeait à ses heures.
Après quelques moments l'appétit vint : l'oiseau,
S'approchant du bord, vit sur l'eau
Des tanches qui sortaient du fond de ces demeures.
Le mets ne lui plut pas, il s'attendait à mieux,
Et montrait un goût dédaigneux
Comme le rat du bon Horace.
Moi, des tanches ! dit-il ; moi, héron, que je fasse
Une si pauvre chère ! Et pour qui me prend-on ?
La tanche rebutée, il trouva du goujon.
Du goujon ! c'est bien là le dîner d'un héron !
J'ouvrirais pour si peu le bec ! aux dieux ne plaise !
Il l'ouvrit pour bien moins : tout alla de façon
Qu'il ne vit plus aucun poisson.
La faim le prit : il fut tout heureux et tout aise
De rencontrer un limaçon.
Ne soyons pas si difficiles :
Les plus accommodants, ce sont les plus habiles ;
On hasarde de perdre en voulant trop gagner.
Gardez-vous de rien dédaigner,
Surtout quand vous avez à peu près votre compte.
Bien des gens y sont pris. Ce n'est pas aux hérons
Que je parle : écoutez, humains, un autre conte :
Vous verrez que chez vous j'ai puisé ces leçons.


LA LAITIÈRE ET LE POT AU LAIT
Perrette, sur sa tête ayant un pot au lait
Bien posé sur un coussinet,
Prétendait arriver sans encombre à la ville.
Légère et court vêtue, elle allait à grands pas,
Ayant mis ce jour-là, pour être plus agile,
Cotillon simple et souliers plats.
Notre laitière ainsi troussée
Comptait déjà dans sa pensée
Tout le prix de son lait ; en employait l'argent ;
Achetait un cent d'œufs ; faisait triple couvée :
La chose allait à bien par son soin diligent.
Il m'est, disait-elle, facile
D'élever des poulets autour de ma maison ;
Le renard sera bien habile
S'il ne m'en laisse assez pour avoir un cochon.
Le porc à s'engraisser coûtera peu de son ;
Il était, quand je l'eus, de grosseur raisonnable :
J'aurai, le revendant, de l'argent bel et bon.
Et qui m'empêchera de mettre en notre étable,
Vu le prix dont il est, une vache et son veau,
Que je verrai sauter au milieu du troupeau ?
Perrette là-dessus saute aussi, transportée :
Le lait tombe ; adieu veau, vache, cochon, couvée.
La dame de ces biens, quittant d'un œil marri
Sa fortune ainsi répandue,
Va s'excuser à son mari,
En grand danger d'être battue.
Le récit en farce en fut fait ;
On l'appela le Pot au lait.
Quel esprit ne bat la campagne ?
Qui ne fait châteaux en Espagne ?
Picrochole, Pyrrhus, la laitière, enfin tous,
Autant les sages que les fous.
Chacun songe en veillant ; il n'est rien de plus doux
Une flatteuse erreur emporte alors nos âmes ;
Tout le bien du monde est à nous,
Tous les honneurs, toutes les femmes.
Quand je suis seul, je fais au plus brave un défi ;
Je m'écarte, je vais détrôner le sophi ;
On m'élit roi, mon peuple m'aime ;
Les diadèmes vont sur ma tête pleuvant :
Quelque accident fait-il que je rentre en moi-même ;
Je suis Gros-Jean comme devant.


LE COCHE ET LA MOUCHE
Dans un chemin montant, sablonneux, malaisé,
Et de tous les côtés au soleil exposé,
Six forts chevaux tiraient un coche.
Femmes, moine, vieillards, tout était descendu :
L'attelage suait, soufflait, était rendu.
Une mouche survient, et des chevaux s'approche,
Prétend les animer par son bourdonnement,
Pique l'un, pique l'autre, et pense à tout moment
Qu'elle fait aller la machine,
S'assied sur le timon, sur le nez du cocher.
Aussitôt que le char chemine,
Et qu'elle voit les gens marcher,
Elle s'en attribue uniquement la gloire,
Va, vient, fait l'empressée : il semble que ce soit
Un sergent de bataille allant en chaque endroit
Faire avancer ses gens et hâter la victoire.
La mouche, en ce commun besoin,
Se plaint qu'elle agit seule, et qu'elle a tout le soin ;
Qu'aucun n'aide aux chevaux à se tirer d'affaire.
Le moine disait son bréviaire :
Il prenait bien son temps ! une femme chantait :
C'était bien de chansons qu'alors il s'agissait !
Dame mouche s'en va chanter à leurs oreilles,
Et fait cent sottises pareilles.
Après bien du travail, le coche arrive au haut.
Respirons maintenant ! dit la mouche aussitôt :
J'ai tant fait que nos gens sont enfin dans la plaine.
Çà, messieurs les chevaux, payez-moi de ma peine.
Ainsi certaines gens, faisant les empressés,
S'introduisent dans les affaires :
Ils font partout les nécessaires,
Et, partout importuns, devraient être chassés.


LE SAVETIER ET LE FINANCIER
Un savetier chantait du matin jusqu'au soir :
C'était merveille de le voir,
Merveille de l'ouïr ; il faisait des passages :
Plus content qu'aucun des sept sages.
Son voisin, au contraire, étant tout cousu d'or,
Chantait peu, dormait moins encor :
C'était un homme de finance.
Si sur le point du jour parfois il sommeillait,
Le savetier alors en chantant l'éveillait ;
Et le financier se plaignait
Que les soins de la Providence
N'eussent pas au marché fait vendre le dormir,
Comme le manger et le boire.
En son hôtel il fait venir
Le chanteur, et lui dit : Or çà, sire Grégoire,
Que gagnez-vous par an ? Par an ! ma foi, monsieur
Dit avec un ton de rieur
Le gaillard savetier, ce n'est point ma manière
De compter de la sorte ; et je n'entasse guère
Un jour sur l'autre : il suffit qu'à la fin
J'attrape le bout de l'année ;
Chaque jour amène son pain. -
Eh bien ! que gagnez-vous, dites-moi, par journée ?
Tantôt plus, tantôt moins : le mal est que toujours
(Et sans cela nos gains seraient assez honnêtes),
Le mal est que dans l'an s'entremêlent des jours
Qu'il faut chômer ; on nous ruine en fêtes :
L'une fait tort à l'autre ; et monsieur le curé
De quelque nouveau saint charge toujours son prône.
Le financier, riant de sa naïveté,
Lui dit : Je vous veux mettre aujourd'hui sur le trône.
Prenez ces cent écus ; gardez-les avec soin,
Pour vous en servir au besoin.
Le savetier crut voir tout l'argent que la terre
Avait depuis plus de cent ans,
Produit pour l'usage des gens.
Il retourne chez lui : dans sa cave il enserre
L'argent, et sa joie à la fois.
Plus de chant : il perdit la voix
Du moment qu'il gagna ce qui cause nos peines.
Le sommeil quitta son logis :
Il eut pour hôtes les soucis,
Les soupçons, les alarmes vaines.
Tout le jour il avait l'œil au guet ; et la nuit,
Si quelque chat faisait du bruit,
Le chat prenait l'argent. À la fin le pauvre homme
S'en courut chez celui qu'il ne réveillait plus :
Rendez-moi, lui dit-il, mes chansons et mon somme ;
Et reprenez vos cent écus.


LES ANIMAUX MALADES DE LA PESTE
Un mal qui répand la terreur,
Mal que le ciel en sa fureur
Inventa pour punir les crimes de la terre,
La peste (puisqu'il faut l'appeler par son nom),
Capable d'enrichir en un jour l'Achéron,
Faisait aux animaux la guerre.
Ils ne mouraient pas tous, mais tous étaient frappés :
On n'en voyait point d'occupés
À chercher le soutien d'une mourante vie ;
Nul mets n'excitait leur envie ;
Ni loups ni renards n'épiaient
La douce et l'innocente proie ;
Les tourterelles se fuyaient :
Plus d'amour, partant plus de joie.
Le lion tint conseil, et dit : Mes chers amis,
Je crois que le ciel a permis
Pour nos péchés cette infortune.
Que le plus coupable de nous
Se sacrifie aux traits du céleste courroux ;
Peut-être il obtiendra la guérison commune.
L'histoire nous apprend qu'en de tels accidents
On fait de pareils dévouements.
Ne nous flattons donc point ; voyons sans indulgence
L'état de notre conscience.
Pour moi, satisfaisant mes appétits gloutons,
J'ai dévoré force moutons.
Que m'avaient-ils fait ? nulle offense ;
Même il m'est arrivé quelquefois de manger
Le berger.
Je me dévouerai donc, s'il le faut : mais je pense
Qu'il est bon que chacun s'accuse ainsi que moi ;
Car on doit souhaiter, selon toute justice,
Que le plus coupable périsse.
Sire, dit le renard, vous êtes trop bon roi ;
Vos scrupules font voir trop de délicatesse.
Eh bien ! manger moutons, canaille, sotte espèce,
Est-ce un péché ? Non, non. Vous leur fîtes, seigneur,
En les croquant, beaucoup d'honneur ;
Et quant au berger, l'on peut dire
Qu'il était digne de tous maux,
Étant de ces gens-là qui sur les animaux
Se font un chimérique empire.
Ainsi dit le renard ; et flatteurs d'applaudir.
On n'osa trop approfondir
Du tigre, ni de l'ours, ni des autres puissances,
Les moins pardonnables offenses :
Tous les gens querelleurs, jusqu'aux simples mâtins,
Au dire de chacun, étaient de petits saints.
L'âne vint à son tour, et dit : J'ai souvenance
Qu'en un pré de moines passant,
La faim, l'occasion, l'herbe tendre, et, je pense,
Quelque diable aussi me poussant,
Je tondis de ce pré la largeur de ma langue ;
Je n'en avais nul droit, puisqu'il faut parler net.
À ces mots, on cria haro sur le baudet.
Un loup, quelque peu clerc, prouva par sa harangue
Qu'il fallait dévouer ce maudit animal,
Ce pelé, ce galeux, d'où venait tout leur mal.
Sa peccadille fut jugée un cas pendable.
Manger l'herbe d'autrui ! quel crime abominable !
Rien que la mort n'était capable
D'expier son forfait. On le lui fit bien voir.
Selon que vous serez puissant ou misérable,
Les jugements de cour vous rendront blanc ou noir.


LA POULE AUX ŒUFS D'OR
L'avarice perd tout en voulant tout gagner.
Je ne veux, pour le témoigner,
Que celui dont la poule, à ce que dit la fable,
Pondait tous les jours un œuf d'or.
Il crut que, dans son corps, elle avait un trésor ;
Il la tua, l'ouvrit, et la trouva semblable
À celle dont les œufs ne lui rapportaient rien,
S'étant lui-même ôté le plus beau de son bien.
Belle leçon pour les gens chiches !
Pendant ces derniers temps combien en a-t-on vus
Qui du soir au matin sont pauvres devenus
Pour vouloir trop tôt être riches !


L'OURS ET LES DEUX COMPAGNONS
Deux compagnons, pressés d'argent,
À leur voisin fourreur vendirent
La peau d'un ours encor vivant,
Mais qu'ils tueraient bientôt, du moins à ce qu'ils dirent,
C'était le roi des ours au compte de ces gens.
Le marchand à sa peau devait faire fortune ;
Elle garantirait des froids les plus cuisants ;
On en pourrait fourrer plutôt deux robes qu'une.
Dindenaut prisait moins ses moutons qu'eux leur ours :
Leur, à leur compte, et non à celui de la bête.
S'offrant de la livrer au plus tard dans deux jours,
Ils conviennent de prix, et se mettent en quête,
Trouvent l'ours qui s'avance et vient vers eux au trot,
Voilà mes gens frappés comme d'un coup de foudre.
Le marché ne tint pas ; il fallut le résoudre :
D'intérêts contre l'ours, on n'en dit pas un mot.
L'un des deux compagnons grimpe au faîte d'un arbre ;
L'autre, plus froid que n'est un marbre,
Se couche sur le nez, fait le mort, tient son vent,
Ayant quelque part ouï dire
Que l'ours s'acharne peu souvent
Sur un corps qui ne vit, ne meut, ni ne respire.
Seigneur ours, comme un sot, donna dans ce panneau :
Il voit ce corps gisant, le croit privé de vie ;
Et, de peur de supercherie,
Le tourne, le retourne, approche son museau,
Flaire aux passages de l'haleine.
C'est, dit-il, un cadavre ; ôtons-nous, car il sent.
À ces mots, l'ours s'en va dans la forêt prochaine.
L'un de nos deux marchands de son arbre descend,
Court à son compagnon, lui dit que c'est merveille
Qu'il n'ait eu seulement que la peur pour tout mal.
Eh bien ! ajouta-t-il, la peau de l'animal ?
Mais que t'a-t-il dit à l'oreille ?
Car il t'approchait de bien près,
Te retournant avec sa serre.
Il m'a dit qu'il ne faut jamais
Vendre la peau de l'ours qu'on ne l'ait mis par terre.


LE RENARD ET LE BOUC
Capitaine renard allait de compagnie
Avec son ami bouc des plus haut encornés :
Celui-ci ne voyait pas plus loin que son nez ;
L'autre était passé maître en fait de tromperie.
La soif les obligea de descendre en un puits ;
Là chacun d'eux se désaltère.
Après qu'abondamment tous deux en eurent pris,
Le renard dit au bouc : Que ferons-nous, compère ?
Ce n'est pas tout de boire, il faut sortir d'ici.
Lève tes pieds en haut, et tes cornes aussi ;
Mets-les contre le mur : le long de ton échine
Je grimperai premièrement ;
Puis sur tes cornes m'élevant,
À l'aide de cette machine,
De ce lieu-ci je sortirai,
Après quoi je t'en tirerai.
Par ma barbe, dit l'autre, il est bon ; et je loue
Les gens bien sensés comme toi.
Je n'aurais jamais, quant à moi,
Trouvé ce secret, je l'avoue.
Le renard sort du puits, laisse son compagnon,
Et vous lui fait un beau sermon
Pour l'exhorter à patience.
Si le ciel t'eût, dit-il, donné par excellence
Autant de jugement que de barbe au menton,
Tu n'aurais pas, à la légère,
Descendu dans ce puits. Or, adieu ; j'en suis hors :
Tâche de t'en tirer et fais tous les efforts ;
Car, pour moi, j'ai certaine affaire
Qui ne me permet pas d'arrêter en chemin.
En toute chose il faut considérer la fin.


LE CERF SE VOYANT DANS L'EAU
Dans le cristal d'une fontaine
Un cerf se mirant autrefois
Louait la beauté de son bois,
Et ne pouvait qu'avecque peine
Souffrir ses jambes de fuseaux,
Dont il voyait l'objet se perdre dans les eaux.
Quelle proportion de mes pieds à ma tête !
Disait-il en voyant leur ombre avec douleur :
Des taillis les plus hauts mon front atteint le faîte ;
Mes pieds ne me font point d'honneur.
Tout en parlant de la sorte,
Un limier le fait partir.
Il tâche à se garantir ;
Dans les forêts il s'emporte :
Son bois, dommageable ornement,
L'arrêtant à chaque moment,
Nuit à l'office que lui rendent
Ses pieds de qui ses jours dépendent.
Il se dédit alors, et maudit les présents
Que le ciel lui fait tous les ans.
Nous faisons cas du beau, nous méprisons l'utile ;
Et le beau souvent nous détruit.
Ce cerf blâme ses pieds qui le rendent agile ;
Il estime un bois qui lui nuit.


LE LOUP DEVENU BERGER
Un loup qui commençait d'avoir petite part
Aux brebis de son voisinage,
Crut qu'il fallait s'aider de la peau du renard,
Et faire un nouveau personnage
Il s'habille en berger, endosse un hoqueton,
Fait sa houlette d'un bâton,
Sans oublier la cornemuse.
Pour pousser jusqu'au bout la ruse,
Il aurait volontiers écrit sur son chapeau :
"C'est moi qui suis Guillot, berger de ce troupeau."
Sa personne étant ainsi faite,
Et ses pieds de devant posés sur sa houlette,
Guillot le sycophante approche doucement.
Guillot, le vrai Guillot, étendu sur l'herbette,
Dormait alors profondément ;
Son chien dormait aussi, comme aussi sa musette :
La plupart des brebis dormaient pareillement.
L'hypocrite les laissa faire ;
Et, pour pouvoir mener vers son fort les brebis,
Il voulut ajouter la parole aux habits,
Chose qu'il croyait nécessaire ;
Mais cela gâta son affaire :
Il ne put du pasteur contrefaire la voix.
Le ton dont il parla fit retentir les bois,
Et découvrit tout le mystère.
Chacun se réveille à ce son,
Les brebis, le chien, le garçon.
Le pauvre loup, dans cet esclandre,
Empêché par son hoqueton,
Ne put ni fuir ni se défendre.
Toujours par quelque endroit fourbes se laissent prendre
Quiconque est loup agisse en loup ;
C'est le plus certain de beaucoup.


LE RAT DE VILLE, ET LE RAT DES CHAMPS
Autrefois le rat de ville
Invita le rat des champs,
D'une façon fort civile,
À des reliefs d'ortolans.
Sur un tapis de Turquie
Le couvert se trouva mis.
Je laisse à penser la vie
Que firent ces deux amis.
Le régal fut fort honnête ;
Rien ne manquait au festin :
Mais quelqu'un troubla la fête
Pendant qu'ils étaient en train.
À la porte de la salle
Ils entendirent du bruit :
Le rat de ville détale ;
Son camarade le suit.
Le bruit cesse, on se retire :
Rats en campagne aussitôt ;
Et le citadin de dire :
Achevons tout notre rôt.
C'est assez, dit le rustique :
Demain vous viendrez chez moi.
Ce n'est pas que je me pique
De tous vos festins de roi :
Mais rien ne vient m'interrompre ;
Je mange tout à loisir.
Adieu donc. Fi du plaisir
Que la crainte peut corrompre !


LE PETIT POISSON ET LE PÊCHEUR
Petit poisson deviendra grand,
Pourvu que Dieu lui prête vie ;
Mais le lâcher en attendant,
Je tiens pour moi que c'est folie,
Car de le rattraper il n'est pas trop certain.
Un carpeau qui n'était encore que fretin,
Fut pris par un pêcheur au bord d'une rivière.
Tout fait nombre, dit l'homme, en voyant son butin ;
Voilà commencement de chère et de festin :
Mettons-le en notre gibecière.
Le pauvre carpillon lui dit en sa manière :
Que ferez-vous de moi ? je ne saurais fournir
Au plus qu'une demi-bouchée.
Laissez-moi carpe devenir :
Je serai par vous repêchée ;
Quelque gros partisan m'achètera bien cher :
Au lieu qu'il vous en faut chercher
Peut-être encor cent de ma taille
Pour faire un plat : quel plat ! croyez-moi, rien qui vaille.
Rien qui vaille ! eh bien ! soit, repartit le pêcheur ;
Poisson, mon bel ami, qui faites le prêcheur,
Vous irez dans la poêle ; et, vous avez beau dire,
Dès ce soir on vous fera frire.
Un Tiens, vaut, ce dit-on, mieux que deux Tu l'auras :
L'un est sûr, l'autre ne l'est pas.


LE POT DE TERRE ET LE POT DE FER
Le pot de fer proposa
Au pot de terre un voyage.
Celui-ci s'en excusa,
Disant qu'il ferait que sage
De garder le coin du feu :
Car il lui fallait si peu,
Si peu que la moindre chose
De son débris serait cause :
Il n'en reviendrait morceau.
Pour vous, dit-il, dont la peau
Est plus dure que la mienne,
Je ne vois rien qui vous tienne.
Nous vous mettrons à couvert,
Repartit le pot de fer :
Si quelque matière dure
Vous menace d'aventure,
Entre deux je passerai,
Et du coup vous sauverai.
Cette offre le persuade.
Pot de fer son camarade
Se met droit à ses côtés.
Mes gens s'en vont à trois pieds
Clopin clopant comme ils peuvent,
L'un contre l'autre jetés
Au moindre hoquet qu'ils treuvent.
Le pot de terre en souffre ; il n'eut pas fait cent pas
Que par son compagnon il fut mis en éclats,
Sans qu'il eût lieu de se plaindre.
Ne nous associons qu'avecque nos égaux ;
Ou bien il nous faudra craindre
Le destin d'un de ces pots.


LE LABOUREUR ET SES ENFANTS
Travaillez, prenez de la peine :
C'est le fonds qui manque le moins.
Un riche laboureur, sentant sa mort prochaine,
Fit venir ses enfants, leur parla sans témoins.
Gardez-vous, leur dit-il, de vendre l'héritage
Que nous ont laissé nos parents :
Un trésor est caché dedans.
Je ne sais pas l'endroit ; mais un peu de courage
Vous le fera trouver : vous en viendrez à bout.
Remuez votre champ dès qu'on aura fait l'oût :
Creusez, fouillez, bêchez ; ne laissez nulle place
Où la main ne passe et repasse.
Le père mort, les fils vous retournent le champ,
De çà, de là, partout ; si bien qu'au bout de l'an
Il en rapporta davantage.
D'argent, point de caché. Mais le père fut sage
De leur montrer, avant sa mort,
Que le travail est un trésor.


LE MEUNIER, SON FILS, ET L'ÂNE
L'invention des arts étant un droit d'aînesse,
Nous devons l'apologue à l'ancienne Grèce :
Mais ce champ ne se peut tellement moissonner
Que les derniers venus n'y trouvent à glaner.
La feinte est un pays plein de terres désertes ;
Tous les jours nos auteurs y font des découvertes.
Je t'en veux dire un trait assez bien inventé :
Autrefois à Racan Malherbe l'a conté.
Ces deux rivaux d'Horace, héritiers de sa lyre,
Disciples d'Apollon, nos maîtres, pour mieux dire,
Se rencontrant un jour tout seuls et sans témoins
(Comme ils se confiaient leurs pensers et leurs soins),
Racan commence ainsi : Dites-moi, je vous prie,
Vous qui devez savoir les choses de la vie,
Qui par tous ses degrés avez déjà passé,
Et que rien ne doit fuir en cet âge avancé,
À quoi me résoudrai-je ? Il est temps que j'y pense.
Vous connaissez mon bien, mon talent, ma naissance :
Dois-je dans la province établir mon séjour,
Prendre emploi dans l'armée, ou bien charge à la cour ?
Tout au monde est mêlé d'amertume et de charmes :
La guerre a ses douceurs, l'hymen a ses alarmes.
Si je suivais mon goût, je saurais où buter ;
Mais j'ai les miens, la cour, le peuple à contenter.
Malherbe là-dessus : Contenter tout le monde !
Écoutez ce récit avant que je réponde.
J'ai lu dans quelque endroit qu'un meunier et son fils,
L'un vieillard, l'autre enfant, non pas des plus petits,
Mais garçon de quinze ans, si j'ai bonne mémoire,
Allaient vendre leur âne, un certain jour de foire.
Afin qu'il fût plus frais et de meilleur débit,
On lui lia les pieds, on vous le suspendit ;
Puis cet homme et son fils le portent comme un lustre.
Pauvres gens ! idiots ! couple ignorant et rustre !
Le premier qui les vit de rire s'éclata :
Quelle farce, dit-il, vont jouer ces gens-là ?
Le plus âne des trois n'est pas celui qu'on pense.
Le meunier, à ces mots, connaît son ignorance ;
Il met sur pieds sa bête, et la fait détaler.
L'âne, qui goûtait fort l'autre façon d'aller,
Se plaint en son patois. Le meunier n'en a cure,
Il fait monter son fils, il suit : et, d'aventure,
Passent trois bons marchands. Cet objet leur déplut.
Le plus vieux au garçon s'écria tant qu'il put :
Oh là ! oh ! descendez, que l'on ne vous le dise,
Jeune homme, qui menez laquais à barbe grise !
C'était à vous de suivre, au vieillard de monter.
Messieurs, dit le meunier, il vous faut contenter.
L'enfant met pied à terre, et puis le vieillard monte ;
Quand trois filles passant, l'une dit : C'est grand'honte
Qu'il faille voir ainsi clocher ce jeune fils,
Tandis que ce nigaud, comme un évêque assis,
Fait le veau sur son âne, et pense être bien sage.
Il n'est, dit le meunier, plus de veaux à mon âge :
Passez votre chemin, la fille, et m'en croyez.
Après maints quolibets coup sur coup renvoyés,
L'homme crut avoir tort, et mit son fils en croupe.
Au bout de trente pas, une troisième troupe
Trouve encore à gloser. L'un dit : Ces gens sont fous !
Le baudet n'en peut plus ; il mourra sous leurs coups.
Eh quoi ! charger ainsi cette pauvre bourrique !
N'ont-ils point de pitié de leur vieux domestique ?
Sans doute qu'à la foire ils vont vendre sa peau.
Parbleu ! dit le meunier, est bien fou du cerveau
Qui prétend contenter tout le monde et son père.
Essayons toutefois si par quelque manière
Nous en viendrons à bout. Ils descendent tous deux.
L'âne se prélassant marche seul devant eux.
Un quidam les rencontre et dit : Est-ce la mode
Que baudet aille à l'aise, et meunier s'incommode ?
Qui de l'âne ou du maître est fait pour se lasser ?
Je conseille à ces gens de le faire enchâsser.
Ils usent leurs souliers, et conservent leur âne !
Nicolas, au rebours : car, quand il va voir Jeanne,
Il monte sur sa bête ; et la chanson le dit.
Beau trio de baudets ! le meunier repartit :
Je suis âne, il est vrai, j'en conviens, je l'avoue ;
Mais que dorénavant on me blâme, on me loue,
Qu'on dise quelque chose ou qu'on ne dise rien,
J'en veux faire à ma tête. Il le fit, et fit bien.
Quant à vous, suivez Mars, ou l'Amour, ou le prince ;
Allez, venez, courez ; demeurez en province ;
Prenez femme, abbaye, emploi, gouvernement :
Les gens en parleront, n'en doutez nullement.


LE CHAT, LA BELETTE, ET LE PETIT LAPIN
Du palais d'un jeune lapin
Dame belette, un beau matin,
S'empara : c'est une rusée.
Le maître étant absent, ce lui fut chose aisée.
Elle porta chez lui ses pénates, un jour
Qu'il était allé faire à l'Aurore sa cour
Parmi le thym et la rosée.
Après qu'il eut brouté, trotté, fait tous ses tours,
Jeannot lapin retourne aux souterrains séjours.
La belette avait mis le nez à la fenêtre.
Ô dieux hospitaliers ! que vois-je ici paraître ?
Dit l'animal chassé du paternel logis.
Holà ! madame la belette,
Que l'on déloge sans trompette,
Ou je vais avertir tous les rats du pays.
La dame au nez pointu répondit que la terre
Était au premier occupant.
C'était un beau sujet de guerre,
Qu'un logis où lui-même il n'entrait qu'en rampant !
Et quand ce serait un royaume,
Je voudrais bien savoir, dit-elle, quelle loi
En a pour toujours fait l'octroi
À Jean, fils ou neveu de Pierre ou de Guillaume,
Plutôt qu'à Paul, plutôt qu'à moi.
Jean lapin allégua la coutume et l'usage :
Ce sont, dit-il, leurs lois qui m'ont de ce logis
Rendu maître et seigneur, et qui, de père en fils,
L'ont de Pierre à Simon, puis à moi Jean, transmis.
Le premier occupant, est-ce une loi plus sage ?
Or bien, sans crier davantage,
Rapportons-nous, dit-elle, à Raminagrobis.
C'était un chat vivant comme un dévot ermite,
Un chat faisant la chattemite,
Un saint homme de chat, bien fourré, gros et gras,
Arbitre expert sur tous les cas.
Jean lapin pour juge l'agrée.
Les voilà tous deux arrivés
Devant sa majesté fourrée.
Grippeminaud leur dit : Mes enfants, approchez,
Approchez : je suis sourd, les ans en sont la cause.
L'un et l'autre approcha, ne craignant nulle chose.
Aussitôt qu'à portée il vit les contestants,
Grippeminaud le bon apôtre,
Jetant des deux côtés la griffe en même temps,
Mit les plaideurs d'accord en croquant l'un et l'autre.
Ceci ressemble fort aux débats qu'ont parfois
Les petits souverains se rapportant aux rois.


LES DEUX MULETS
Deux mulets cheminaient, l'un d'avoine chargé,
L'autre portant l'argent de la gabelle.
Celui-ci, glorieux d'une charge si belle,
N'eût voulu pour beaucoup en être soulagé.
Il marchait d'un pas relevé
Et faisait sonner sa sonnette ;
Quand l'ennemi se présentant,
Comme il en voulait à l'argent,
Sur le mulet du fisc une troupe se jette,
Le saisit au frein, et l'arrête.
Le mulet, en se défendant,
Se sent percer de coups ; il gémit, il soupire.
Est-ce donc là, dit-il, ce qu'on m'avait promis ?
Ce mulet qui me suit du danger se retire,
Et moi j'y tombe et je péris !
Ami, lui dit son camarade,
Il n'est pas toujours bon d'avoir un haut emploi :
Si tu n'avais servi qu'un meunier comme moi,
Tu ne serais pas si malade.
"""

print(f"{len(corpus)} caractères")
print(corpus[:236])

In [ ]:
chars = sorted(set(corpus))              # les caractères distincts, triés
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}   # caractère -> numéro
itos = {i: c for c, i in stoi.items()}       # numéro -> caractère

data = torch.tensor([stoi[c] for c in corpus])
print(f"vocabulaire : {vocab_size} caractères | corpus : {data.shape[0]} tokens")

In [ ]:
def decouper_en_tetes(X, n_heads):
    """(B, T, d_model) -> (B, n_heads, T, d_k), avec d_k = d_model // n_heads."""
    B, T, d_model = X.shape
    d_k = d_model // n_heads
    return X.view(B, T, n_heads, d_k).transpose(1, 2)


class MultiHeadAttention(nn.Module):
    """n_heads attentions en parallèle dans des sous-espaces de dimension d_k."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model doit être divisible par n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, d_model = x.shape
        Q = decouper_en_tetes(self.W_Q(x), self.n_heads)
        K = decouper_en_tetes(self.W_K(x), self.n_heads)
        V = decouper_en_tetes(self.W_V(x), self.n_heads)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float('-inf'))
        poids = torch.softmax(scores, dim=-1)
        melange = poids @ V
        out = melange.transpose(1, 2).contiguous().view(B, T, d_model)
        return self.W_O(out)


class FeedForward(nn.Module):
    """Le MLP du chapitre 6, appliqué à chaque position indépendamment."""

    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Attention + FFN, chacun avec Pre-LN et résidu (le bloc du chapitre 10)."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x

In [ ]:
block_size = 64
d_model, n_heads, n_layers, d_ff = 96, 4, 2, 384


class GPT(nn.Module):
    """Embeddings + positions -> n_layers blocs -> LayerNorm -> tête de sortie."""

    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))
        for bloc in self.blocs:
            h = bloc(h, self.masque)
        return self.tete(self.ln_final(h))

In [ ]:
def fabriquer_batch(taille=32):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y


gpt = GPT()
print(f"GPT : {sum(p.numel() for p in gpt.parameters())} paramètres")

optimiseur = torch.optim.AdamW(gpt.parameters(), lr=3e-3)
debut = time.time()
for step in range(1501):
    x, y = fabriquer_batch()
    logits = gpt(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    optimiseur.zero_grad()
    loss.backward()
    optimiseur.step()
    if step % 500 == 0:
        print(f"étape {step:5d} | loss {loss.item():.3f}")
print(f"Ré-entraînement terminé en {time.time() - debut:.0f} s")

### L'étalon de fluidité : la perplexité fables

Avant de toucher au modèle, on fige un thermomètre. Si un entraînement de préférences casse le français du modèle, ce chiffre le dira.

In [ ]:
# Notre étalon de fluidité : la perplexité sur 64 fenêtres de fables, FIGÉES une
# fois pour toutes. Attention au vocabulaire du chapitre 16 : ces fenêtres viennent
# du corpus d'ENTRAÎNEMENT, ce n'est donc pas une mesure de généralisation. Ici on
# s'en sert pour autre chose : détecter si un entraînement ultérieur CASSE le
# français que le modèle savait déjà écrire. Pour ça, le train suffit.
torch.manual_seed(7)
eval_ix = torch.randint(0, len(data) - block_size - 1, (64,))
eval_x = torch.stack([data[i : i + block_size] for i in eval_ix])
eval_y = torch.stack([data[i + 1 : i + block_size + 1] for i in eval_ix])


def perplexite_fables(model):
    """exp(cross-entropy moyenne) sur les 64 fenêtres figées (chapitre 16)."""
    model.eval()
    with torch.no_grad():
        logits = model(eval_x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), eval_y.view(-1))
    model.train()
    return math.exp(loss.item())


ppl_sft = perplexite_fables(gpt)
print(f"perplexité fables (modèle de départ) : {ppl_sft:.2f}")

In [ ]:
def generer(model, prompt="\n", longueur=60, temperature=0.8, graine=3):
    """Écrit `longueur` caractères à la suite de `prompt` (génération du ch. 10)."""
    torch.manual_seed(graine)                    # même graine = sortie comparable
    model.eval()
    ctx = ([stoi["\n"]] * block_size + [stoi[c] for c in prompt])[-block_size:]
    sortie = []
    with torch.no_grad():
        for _ in range(longueur):
            logits = model(torch.tensor([ctx]))[:, -1, :]
            probas = torch.softmax(logits / temperature, dim=-1)
            i = torch.multinomial(probas, num_samples=1).item()
            sortie.append(itos[i])
            ctx = ctx[1:] + [i]
    model.train()
    return prompt + "".join(sortie)


print(repr(generer(gpt, "Le corbeau ")))

## 2. Le dataset de préférences, fabriqué à la main

Trente-deux paires `(prompt, chosen, rejected)` sur l'univers des fables. Les deux réponses sont du français correct : la `chosen` raconte quelque chose, la `rejected` est paresseuse. C'est exactement le genre de nuance que la loss du chapitre 17 ne voit pas.

In [ ]:
# Les rejected : huit fins de phrase paresseuses, en français correct.
paresseuses = ["s'en alla.\n", "ne dit rien.\n", "était là.\n", "fut bien aise.\n",
               "ne fit rien.\n", "resta là.\n", "s'en fut.\n", "dit : bon.\n"]

# Les chosen : trente-deux suites vivantes, dans l'esprit des fables.
vivantes = [
    ("Le corbeau ", "tenait en son bec un beau fromage.\n"),
    ("La cigale ", "chanta tout l'été sans rien garder.\n"),
    ("Maître renard ", "flatta le corbeau pour son fromage.\n"),
    ("Le lion ", "régnait terrible sur les animaux.\n"),
    ("La fourmi ", "refusa de prêter un seul grain.\n"),
    ("Le loup ", "chercha querelle au petit agneau.\n"),
    ("Un agneau ", "buvait au courant d'une onde pure.\n"),
    ("Le renard ", "loua le plumage du corbeau.\n"),
    ("La grenouille ", "voulut égaler le boeuf en grosseur.\n"),
    ("Le rat des villes ", "convia le rat des champs au festin.\n"),
    ("Le chêne ", "se vantait de braver la tempête.\n"),
    ("Le laboureur ", "montra le trésor caché dans le champ.\n"),
    ("La tortue ", "gagna la course contre le lièvre.\n"),
    ("Le berger ", "cria au loup une fois de trop.\n"),
    ("Le meunier ", "suivit tous les avis et perdit l'âne.\n"),
    ("La belette ", "se trouva prise au piège du grenier.\n"),
    ("Le chat ", "jugea l'affaire en croquant les deux.\n"),
    ("Le héron ", "dédaigna la carpe et prit un limaçon.\n"),
    ("Le pêcheur ", "rejeta le petit poisson à l'eau.\n"),
    ("Le chien ", "montra son cou pelé par le collier.\n"),
    ("La colombe ", "sauva la fourmi qui se noyait.\n"),
    ("Le vieillard ", "planta pour les enfants à venir.\n"),
    ("Le singe ", "fit rire la cour par ses grimaces.\n"),
    ("L'âne ", "se chargea d'éponges et se noya.\n"),
    ("Le rossignol ", "chanta pour sauver sa vie au faucon.\n"),
    ("La brebis ", "paya pour le crime qu'elle n'eut pas.\n"),
    ("Le paon ", "se plaignait de n'avoir pas de voix.\n"),
    ("Le moucheron ", "vainquit le lion et périt d'une toile.\n"),
    ("La poule ", "aux oeufs d'or mourut de l'avarice.\n"),
    ("Le cerf ", "se prit les bois dans les branches.\n"),
    ("Le serpent ", "mordit la main qui le réchauffait.\n"),
    ("Le lièvre ", "perdit la course à force de dormir.\n"),
]

paires = [
    {"prompt": pr, "chosen": ch, "rejected": paresseuses[i % len(paresseuses)]}
    for i, (pr, ch) in enumerate(vivantes)
]

assert all(len(p["prompt"]) + len(p["chosen"]) <= block_size for p in paires), \
    "chaque prompt + réponse doit tenir dans les 64 caractères de contexte"
assert all(c in stoi for p in paires for c in p["prompt"] + p["chosen"] + p["rejected"]), \
    "tous les caractères doivent exister dans le vocabulaire des fables"

print(f"{len(paires)} paires | exemple :")
print("  prompt   :", repr(paires[0]["prompt"]))
print("  chosen   :", repr(paires[0]["chosen"]))
print("  rejected :", repr(paires[0]["rejected"]))

## 3. TODO 1 · La log-prob d'une réponse

La brique de base de DPO : `log p(réponse | prompt)`, lue en une seule passe avant. Réflexe shape : `logits` est `(1, L-1, vocab_size)`, `gather` en tire une log-prob par caractère prédit `(L-1,)`, et on ne somme que la tranche de la réponse.

In [ ]:
def logprob_reponse(model, prompt, reponse):
    """log p(reponse | prompt) : la somme des log-probs des caractères de la réponse.

    Étapes : encoder prompt + réponse, faire UNE passe avant, prendre le log-softmax,
    lire la log-prob du vrai caractère suivant à chaque position (gather), puis ne
    sommer que sur les positions de la réponse (le prompt n'est pas noté).
    """
    ids_prompt = [stoi[c] for c in prompt]
    ids_reponse = [stoi[c] for c in reponse]
    ids = torch.tensor([ids_prompt + ids_reponse])          # (1, L)

    logits = model(ids[:, :-1])                             # (1, L-1, vocab_size)
    logp = F.log_softmax(logits, dim=-1)                    # scores -> log-probs

    cibles = ids[:, 1:]                                     # (1, L-1) le "caractère suivant"
    # TODO(toi) : lis la log-prob de chaque cible avec logp.gather(-1, ...),
    # tu dois obtenir un tenseur (L-1,) : une log-prob par caractère prédit.
    logp_par_car = ...

    # TODO(toi) : somme UNIQUEMENT les len(ids_reponse) dernières valeurs
    # (les cibles de la réponse ; le prompt n'est pas noté).
    return ...

In [ ]:
# Vérification contre un calcul naïf, caractère par caractère.
p0 = paires[0]
with torch.no_grad():
    lp = logprob_reponse(gpt, p0["prompt"], p0["chosen"])

assert isinstance(lp, torch.Tensor) and lp.dim() == 0, "remplace les ... : on attend un scalaire"

with torch.no_grad():
    ids = [stoi[c] for c in p0["prompt"] + p0["chosen"]]
    naive = 0.0
    for t in range(len(p0["prompt"]), len(ids)):             # chaque caractère de la réponse
        logits = gpt(torch.tensor([ids[:t]]))[0, -1]         # prédit à partir du préfixe
        naive += F.log_softmax(logits, dim=-1)[ids[t]].item()

assert abs(lp.item() - naive) < 1e-3, f"{lp.item():.4f} vs naïf {naive:.4f} : mauvaise tranche ?"
print(f"TODO 1 validé : log p(chosen | prompt) = {lp.item():.1f} (naïf : {naive:.1f})")

### La surprise mesurée : la paresseuse gagne

Sur nos 32 paires, la réponse courte et creuse est presque toujours la plus probable : moins de caractères, donc moins de log-probs négatives à additionner. Le modèle du chapitre 17 obéit, mais il n'a aucune raison de viser la réponse *utile*.

In [ ]:
# La mesure qui motive tout le chapitre : lequel des deux textes notre
# modèle trouve-t-il le plus probable, sur chacune des 32 paires ?
with torch.no_grad():
    nb_bon_sens = sum(
        1 for p in paires
        if logprob_reponse(gpt, p["prompt"], p["chosen"])
        > logprob_reponse(gpt, p["prompt"], p["rejected"])
    )
    lp_c = logprob_reponse(gpt, p0["prompt"], p0["chosen"]).item()
    lp_r = logprob_reponse(gpt, p0["prompt"], p0["rejected"]).item()

print(f"paires où chosen est plus probable que rejected : {nb_bon_sens}/{len(paires)}")
print(f"exemple (Le corbeau) : chosen {lp_c:.1f} | rejected {lp_r:.1f}")
assert nb_bon_sens <= len(paires) // 2, "surprise attendue : la paresseuse gagne presque toujours"

## 4. TODO 2 · La loss DPO

Quatre log-probs entrent, un scalaire sort. Deux soustractions (les log-ratios), une soustraction (la marge), une sigmoïde en log. C'est tout DPO.

In [ ]:
def loss_dpo(lp_chosen, lp_rejected, ref_chosen, ref_rejected, beta=0.1):
    """La perte DPO pour une paire (ou un batch de paires).

    1. le log-ratio de la policy   : combien ELLE préfère chosen à rejected
    2. le log-ratio de la référence : la même préférence, chez le modèle gelé
    3. la marge : ce que la policy a appris EN PLUS de la référence
    4. -logsigmoid : une cross-entropy binaire sur cette marge
    """
    # TODO(toi) : les deux log-ratios (une soustraction chacun, log a - log b = log(a/b))
    ratio_policy = ...
    ratio_ref = ...
    # TODO(toi) : la marge, puis -F.logsigmoid(beta * marge).mean()
    return ...

In [ ]:
z = torch.zeros(1)
un = torch.ones(1)

# Marge nulle (policy = référence) : la loss doit valoir ln 2, le point neutre.
l_neutre = loss_dpo(z, z, z, z)
assert abs(l_neutre.item() - math.log(2)) < 1e-6, "à marge nulle, -log sigmoid(0) = ln 2"

# Marge positive (la policy pousse chosen dans le bon sens) : la loss doit baisser.
l_bonne = loss_dpo(un * 2, z, z, z)          # la policy préfère chosen de +2
l_mauvaise = loss_dpo(z, un * 2, z, z)       # la policy préfère rejected de +2
assert l_bonne < l_neutre < l_mauvaise, "la loss doit récompenser la marge positive"

# beta amplifie la marge avant la sigmoïde.
assert loss_dpo(un * 2, z, z, z, beta=0.5) < loss_dpo(un * 2, z, z, z, beta=0.1)

print(f"TODO 2 validé : loss à marge nulle = {l_neutre.item():.4f} (ln 2 = {math.log(2):.4f})")
print(f"même paire, marge +2 : beta=0.1 -> {loss_dpo(un*2, z, z, z, beta=0.1).item():.4f}"
      f" | beta=0.5 -> {loss_dpo(un*2, z, z, z, beta=0.5).item():.4f}")

## 5. TODO 3 · L'entraînement DPO, pour de vrai

Deux copies du modèle : la **policy** (entraînée) et la **référence** (gelée). À chaque paire : 4 log-probs, une loss, un pas d'optimiseur.

In [ ]:
# La référence : une COPIE GELÉE du modèle de départ. C'est l'ancre de DPO.
reference = copy.deepcopy(gpt)
for param in reference.parameters():
    param.requires_grad_(False)
reference.eval()

# La policy : la copie qu'on va entraîner (le modèle de départ reste intact).
policy = copy.deepcopy(gpt)
print("policy et référence prêtes : deux copies du même modèle, une seule apprendra")

In [ ]:
def entrainer_dpo(policy, reference, beta=0.1, epoques=5, lr=5e-5,
                  sans_reference=False, graine=123, verbose=False):
    """La boucle DPO : une paire à la fois, 4 log-probs, une loss, un pas."""
    torch.manual_seed(graine)
    random.seed(graine)
    optimiseur = torch.optim.AdamW(policy.parameters(), lr=lr)
    historique = []
    for epoque in range(epoques):
        ordre = list(range(len(paires)))
        random.shuffle(ordre)
        total = 0.0
        for i in ordre:
            p = paires[i]
            # côté policy : avec gradient (c'est elle qu'on entraîne)
            lp_c = logprob_reponse(policy, p["prompt"], p["chosen"])
            lp_r = logprob_reponse(policy, p["prompt"], p["rejected"])
            if sans_reference:
                # variante MUTILÉE pour le cas qui échoue : pas d'ancre
                loss = -F.logsigmoid(beta * (lp_c - lp_r))
            else:
                # côté référence : gelée, donc sans gradient
                with torch.no_grad():
                    ref_c = logprob_reponse(reference, p["prompt"], p["chosen"])
                    ref_r = logprob_reponse(reference, p["prompt"], p["rejected"])
                # TODO(toi) : appelle loss_dpo avec les 4 log-probs et beta
                loss = ...
            optimiseur.zero_grad()
            loss.backward()
            optimiseur.step()
            total += loss.item()
        historique.append(total / len(paires))
        if verbose:
            print(f"époque {epoque + 1} | loss DPO moyenne {historique[-1]:.4f}")
    return historique

In [ ]:
debut = time.time()
historique = entrainer_dpo(policy, reference, beta=0.1, epoques=5, lr=5e-5, verbose=True)
print(f"Entraînement DPO terminé en {time.time() - debut:.0f} s")
assert historique[-1] < historique[0], "la loss DPO doit descendre"

### Le bilan chiffré : la marge s'ouvre, le français tient

In [ ]:
def bilan(model, reference):
    """Marge moyenne, reward accuracy, préférence brute et perplexité fables."""
    with torch.no_grad():
        marges = []
        bon_sens = 0
        for p in paires:
            lp_c = logprob_reponse(model, p["prompt"], p["chosen"])
            lp_r = logprob_reponse(model, p["prompt"], p["rejected"])
            ref_c = logprob_reponse(reference, p["prompt"], p["chosen"])
            ref_r = logprob_reponse(reference, p["prompt"], p["rejected"])
            marges.append(((lp_c - lp_r) - (ref_c - ref_r)).item())
            if lp_c > lp_r:
                bon_sens += 1
    return {
        "marge": statistics.mean(marges),
        "reward_acc": sum(1 for m in marges if m > 0) / len(marges),
        "pref_brute": bon_sens / len(paires),
        "ppl": perplexite_fables(model),
    }


avant = bilan(gpt, reference)
apres = bilan(policy, reference)
print(f"avant DPO : marge {avant['marge']:6.2f} | reward acc {avant['reward_acc']:4.0%}"
      f" | chosen gagne {avant['pref_brute']:4.0%} | perplexité {avant['ppl']:.2f}")
print(f"après DPO : marge {apres['marge']:6.2f} | reward acc {apres['reward_acc']:4.0%}"
      f" | chosen gagne {apres['pref_brute']:4.0%} | perplexité {apres['ppl']:.2f}")

assert apres["reward_acc"] > 0.9, "presque toutes les marges doivent être positives"
assert apres["ppl"] < avant["ppl"] * 1.10, "la perplexité fables ne doit presque pas bouger"

In [ ]:
print("log-probs totales (policy) avant -> après, sur quatre paires :")
with torch.no_grad():
    for k in [0, 1, 4, 13]:
        p = paires[k]
        av_c = logprob_reponse(gpt, p["prompt"], p["chosen"]).item()
        av_r = logprob_reponse(gpt, p["prompt"], p["rejected"]).item()
        ap_c = logprob_reponse(policy, p["prompt"], p["chosen"]).item()
        ap_r = logprob_reponse(policy, p["prompt"], p["rejected"]).item()
        fleche = "inversée !" if (av_c < av_r and ap_c > ap_r) else "resserrée"
        print(f"  {p['prompt']:<20s} chosen {av_c:6.1f} -> {ap_c:6.1f} | "
              f"rejected {av_r:6.1f} -> {ap_r:6.1f} | {fleche}")

### Générations avant / après

À cette échelle, la génération libre bouge peu : c'est dans les log-probs que la préférence se lit. Les grands modèles, eux, ont assez de capacité pour que la différence s'entende.

In [ ]:
for prompt, graine in [("Le corbeau ", 3), ("La fourmi ", 11)]:
    print("avant :", repr(generer(gpt, prompt, graine=graine)))
    print("après :", repr(generer(policy, prompt, graine=graine)))
    print()

## 6. L'ablation beta

Même recette, quatre valeurs de beta. Petit beta : la policy bouge beaucoup (grosse marge, perplexité qui frémit). Grand beta : le garde-fou serre, tout bouge à peine.

In [ ]:
print("beta  | marge   | reward acc | perplexité fables")
for beta in [0.05, 0.1, 0.5, 2.0]:
    essai = copy.deepcopy(gpt)
    entrainer_dpo(essai, reference, beta=beta, epoques=5, lr=1e-4)
    b = bilan(essai, reference)
    print(f"{beta:4}  | {b['marge']:6.1f}  |    {b['reward_acc']:4.0%}    | {b['ppl']:.2f}")

## 7. Cas qui échoue n°1 · DPO sans modèle de référence

On mutile la loss : plus d'ancre. Le code tourne, toutes les paires finissent gagnées... et la perplexité fables se dégrade nettement. Le modèle paie sa victoire en abîmant sa langue.

In [ ]:
# Cas qui échoue n°1 : on retire la référence de la loss. Le code tourne,
# la loss descend, les paires sont toutes gagnées... et le modèle dérive.
derive = copy.deepcopy(gpt)
entrainer_dpo(derive, reference, beta=0.1, epoques=40, lr=1e-4, sans_reference=True)
b = bilan(derive, reference)
print(f"sans référence, 40 époques : chosen gagne {b['pref_brute']:.0%}"
      f" | marge {b['marge']:.1f} | perplexité {ppl_sft:.2f} -> {b['ppl']:.2f}")
assert b["ppl"] > apres["ppl"], "sans ancre, la perplexité doit se dégrader davantage"

## 8. Cas qui échoue n°2 · La sur-optimisation

Cette fois on garde la référence, mais on insiste beaucoup trop fort. La perplexité explose et la génération le montre : le modèle a gagné le match des paires en cassant son français.

In [ ]:
# Cas qui échoue n°2 : la sur-optimisation. On garde la référence mais on
# insiste : 40 époques, learning rate 20 fois trop grand. Résultat mesuré.
suroptim = copy.deepcopy(gpt)
entrainer_dpo(suroptim, reference, beta=0.1, epoques=40, lr=1e-3)
b2 = bilan(suroptim, reference)
print(f"sur-optimisé : chosen gagne {b2['pref_brute']:.0%}"
      f" | perplexité {ppl_sft:.2f} -> {b2['ppl']:.2f}")
print("génération :", repr(generer(suroptim, "Le corbeau ")))
assert b2["ppl"] > 1.3 * ppl_sft, "la perplexité doit exploser : le modèle a cassé son français"

## Ce que tu viens de faire

- Mesurer qu'un modèle qui obéit ne sait pas préférer : sur 32 paires, la réponse paresseuse était presque toujours la plus probable.
- Implémenter la loss DPO en quatre lignes et l'entraîner pour de vrai : marge ouverte, reward accuracy à 100 %, perplexité intacte.
- Vérifier de tes mains le rôle du modèle de référence : sans lui (ou en sur-optimisant), le modèle gagne les paires et perd sa langue.

Au chapitre 19, on arrête de reconstruire : on lit ce qui a changé depuis GPT-2, avec les yeux de quelqu'un qui a tout construit.